### Chicago 2025 World Championships Race Analysis

This notebook contains an analysis of the 2025 World Championships in Chicago. In this first section, we load the data and get our first look at its characteristics.

In [ ]:
import pandas as pd
from pathlib import Path
import pyrox.models as models

# the gender to analyze
GENDER = "women"
# random state
RANDOM_STATE = 1337
# top-k
TOP_K = 256

# divisions for the analysis
DIV_ELITE, DIV_NON = (
    (models.DivisionName.ELITE_MEN, models.DivisionName.PRO_MEN)
    if GENDER == "men"
    else (models.DivisionName.ELITE_WOMEN, models.DivisionName.PRO_WOMEN)
)

In [ ]:
data_path = Path.cwd() / ".." / ".." / "data" / f"results_{GENDER}.csv"
if not data_path.is_file():
    raise ValueError(f"results not found at path {data_path}")

df = pd.read_csv(data_path)
df.head()

In [ ]:
# exclude values where we couldn't scrape splits
df = df[df["has_splits"]]
print(f"{len(df)} results after filtering those without splits")

In [ ]:
import pyrox.models as models

# number of elites
n_elite = (df["division_name"] == str(DIV_ELITE)).sum()
print(f"{n_elite} elite {GENDER}")

# number of solo pro
n_non = (df["division_name"] == str(DIV_NON)).sum()
print(f"{n_non} non-elite pro solo {GENDER}")

**Distributing the Roxzone**

The elites have no roxzone times recorded - this is just a distinction in how results are recorded between elite and non-elite divisions. We need to allocate this time somewhere to make the analysis valid between the two groups.

In [ ]:
# the elites have no roxzone times
df[df["division_name"] == str(DIV_ELITE)]["roxzone"]

In [ ]:
# the roxzone allocation for each athlete,
# based on total roxzone time evenly distributed across all 8 runs
allocation = (df["roxzone"] / 8).astype(int)
for i in range(8):
    df[f"run_{i+1}"] = df[f"run_{i+1}"] + allocation

In [ ]:
# tolerance in seconds
_TOLERANCE = 10

for i, row in enumerate(df.itertuples()):
    # the total time as a sum of runs and stations
    total = sum([getattr(row, f"run_{i+1}") for i in range(8)]) + sum(
        [getattr(row, str(s)) for s in models.Station]
    )
    # the difference with the athletes official finish time
    diff = abs(row.finish_time - total)  # type: ignore
    if diff > _TOLERANCE:
        raise ValueError(f"exceeded tolerance on row {i}, {diff}s")

**Pace Analysis - Preparation**

In this analysis, we look at the mean and variance of run and station duration for elites and non-elites, and compare the results between the two groups.

In [ ]:
# a dataframe of just the elites
elites = df[df["division_name"] == str(DIV_ELITE)]
# a dataframe of the just the non-elites, limited to top TOP_K finishers
non_elites = df[df["division_name"] == str(DIV_NON)].head(TOP_K)

In [ ]:
import numpy as np

# the column names for the runs
run_cols = [f"run_{i+1}" for i in range(8)]
# the column names for the stations
station_cols = [str(station) for station in models.Station]

# the elite run splits as numpy array
elite_runs = elites[run_cols].to_numpy()
# the elite stations as numpy array
elite_stations = elites[station_cols].to_numpy()

# the non-elite run splits as numpy array
non_runs = non_elites[run_cols].to_numpy()
# the non-elite stations as numpy array
non_stations = non_elites[station_cols].to_numpy()

assert all(
    a.shape[1] == 8 for a in [elite_runs, elite_stations, non_runs, non_stations]
)


# finally, we concatenate to combine all events for both groups;
# e.g. run 1, run 2, ... run 8, ski, sled push, etc...
elites_np = np.concatenate((elite_runs, elite_stations), axis=1)
non_np = np.concatenate((non_runs, non_stations), axis=1)

assert all(a.shape[1] == 16 for a in [elites_np, non_np])

In [ ]:
import numpy as np
import numpy.typing as npt


def mean_and_std(a: npt.NDArray) -> tuple[npt.NDArray, npt.NDArray]:
    """Compute column-wise mean and variance."""
    return np.mean(a, axis=0), np.std(a, axis=0)


# compute mean and standard deviation for elite and non-elite races
elite_means, elite_stds = mean_and_std(elites_np)
non_means, non_stds = mean_and_std(non_np)

In [ ]:
# now, interleave to get events for groups in race order;
# e.g. run 1, ski, run 2, sled push, etc.
from typing import Any, Iterable


def interleave(a: Iterable[Any], b: Iterable[Any]) -> list[Any]:
    return [item for pair in zip(a, b) for item in pair]


# means race order, elite and stds race order, elite
mro_e, sro_e = np.array(interleave(elite_means[:8], elite_means[8:])), np.array(
    interleave(elite_stds[:8], elite_stds[8:])
)
# means race order, non and stds race order, non
mro_n, sro_n = np.array(interleave(non_means[:8], non_means[8:])), np.array(
    interleave(non_stds[:8], non_stds[8:])
)

**Pace Analysis - Some Descriptive Statistics**

We can look at some basic descriptive statistics of the pace analysis for elites and non-elites to see where the time differences are coming from.

In [ ]:
import humanize
from datetime import timedelta

# total finish times
total_e = elites["finish_time"].mean()
total_n = non_elites["finish_time"].mean()

# run mean total duration for each group
run_e = elite_runs.sum(axis=1).mean()
run_n = non_runs.sum(axis=1).mean()

# station mean total duration for each group
station_e = elite_stations.sum(axis=1).mean()
station_n = non_stations.sum(axis=1).mean()

print(f"finish time")
print(f"  elite: {humanize.precisedelta(timedelta(seconds=int(total_e)))}")
print(f"  non-elite: {humanize.precisedelta(timedelta(seconds=int(total_n)))}")
print(
    f"  diff: +{humanize.precisedelta(timedelta(seconds=int(total_e)) - timedelta(seconds=int(total_n)))}"
)

print(f"run total")
print(f"  elite: {humanize.precisedelta(timedelta(seconds=int(run_e)))}")
print(f"  non-elite: {humanize.precisedelta(timedelta(seconds=int(run_n)))}")
print(
    f"  diff: +{humanize.precisedelta(timedelta(seconds=int(run_e)) - timedelta(seconds=int(run_n)))}"
)

print(f"station total")
print(f"  elite: {humanize.precisedelta(timedelta(seconds=int(station_e)))}")
print(f"  non-elite: {humanize.precisedelta(timedelta(seconds=int(station_n)))}")
print(
    f"  diff: +{humanize.precisedelta(timedelta(seconds=int(station_e)) - timedelta(seconds=int(station_n)))}"
)

**Pace Analysis - Comprehensive Plot**

A plot of the pace analysis data gives us a full picture of the whole race for both elites and non-elites.

In [ ]:
from plotnine import (
    ggplot,
    aes,
    element_text,
    labs,
    theme,
    position_dodge,
    geom_segment,
    geom_point,
    scale_y_continuous,
)
from typing import Any


def interleave(a: list[Any], b: list[Any]) -> list[Any]:
    assert len(a) == len(b), "broken precondition"
    return [item for pair in zip(a, b) for item in pair]


def format_seconds_to_mmss(breaks):
    return [f"{int(s//60)}:{int(s%60):02d}" for s in breaks]


def make_plot(
    elite_means: npt.NDArray,
    elite_stds: npt.NDArray,
    non_means: npt.NDArray,
    non_stds: npt.NDArray,
) -> ggplot:
    """Make a plot."""
    # generate the final order of labels
    labels = interleave(
        [f"run_{i+1}" for i in range(8)], [str(station) for station in models.Station]
    )
    # make the labels pretty
    labels = [" ".join(map(str.capitalize, e.split("_"))) for e in labels]

    # compute lower and upper for elites and non-elites
    elite_lower = [int(mean - std) for mean, std in zip(elite_means, elite_stds)]
    elite_upper = [int(mean + std) for mean, std in zip(elite_means, elite_stds)]
    non_lower = [int(mean - std) for mean, std in zip(non_means, non_stds)]
    non_upper = [int(mean + std) for mean, std in zip(non_means, non_stds)]

    # interleave to get final data for df
    mean = interleave(
        [int(v) for v in elite_means.astype(int)],
        [int(v) for v in non_means.astype(int)],
    )
    lower = interleave([v for v in elite_lower], [v for v in non_lower])
    upper = interleave([v for v in elite_upper], [v for v in non_upper])

    df = pd.DataFrame(
        {
            # interleave with self to make 2x copies of each event
            "event": interleave(labels, labels),
            "division": ["elite", "non-elite"] * len(elite_means),
            "mean": mean,
            "lower": lower,
            "upper": upper,
        }
    )
    df["event"] = pd.Categorical(df["event"], categories=labels, ordered=True)

    # dodge so classes are side-by-side instead of overlapping
    d = position_dodge(width=0.4)

    p = (
        ggplot(df, aes("event", "mean", color="division"))
        # vertical line: lower -> upper
        + geom_segment(
            aes(x="event", xend="event", y="lower", yend="upper"), position=d
        )
        # horizontal whisker (lower)
        + geom_segment(
            aes(x="event", xend="event", y="lower", yend="lower"), position=d, size=2
        )
        # horizontal whisker (upper)
        + geom_segment(
            aes(x="event", xend="event", y="upper", yend="upper"), position=d, size=2
        )
        # mean point
        + geom_point(position=d, size=3)
        + theme(figure_size=(14, 6), axis_text_x=element_text(rotation=45, ha="right"))
        + labs(
            x="Event",
            y="Duration (mm:ss)",
            title=f"Elites versus Top-{TOP_K} Non-Elite Pro {GENDER.capitalize()}, Chicago 2025",
        )
        + scale_y_continuous(labels=format_seconds_to_mmss)
    )
    return p


make_plot(mro_e, sro_e, mro_n, sro_n)

In [ ]:
# labels in race order
labels = interleave([f"run_{i+1}" for i in range(8)], [str(s) for s in models.Station])

joined_e = sorted(
    [(label, int(duration)) for label, duration in zip(labels, sro_e)],
    key=lambda p: p[1],
    reverse=True,
)
joined_n = sorted(
    [(label, int(duration)) for label, duration in zip(labels, sro_n)],
    key=lambda p: p[1],
    reverse=True,
)

print(f"elites:")
print(joined_e)

print("non-elites:")
print(joined_n)

**Predicting Elite Performance**

Using logistic regression to identify the race characteristics that predict elite performance.

In [ ]:
np.random.seed(RANDOM_STATE)

# add a row that indicates class (1 = elite, 0 = non-elite)
e = np.hstack((elites_np, np.array([1 for _ in range(len(elites_np))]).reshape(-1, 1)))
n = np.hstack((non_np, np.array([0 for _ in range(len(non_np))]).reshape(-1, 1)))

dataset = np.vstack((e, n))
np.random.shuffle(dataset)

dataset.shape

In [ ]:
# split into samples and classes
X = dataset[:, [i for i in range(16)]]
y = dataset[:, 16]

# create a train / test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

In [ ]:
from sklearn.linear_model import LogisticRegression

# fit the model
clf = LogisticRegression(random_state=RANDOM_STATE)
clf.fit(X_train, y_train)

# score accuracy on held-out examples
acc = clf.score(X_test, y_test)
print(f"accuracy = {acc:.2f}")

In [ ]:
import math

# labels here are not in race order, they are runs, stations
labels = [f"run_{i+1}" for i in range(8)] + [str(s) for s in models.Station]
coeffs = [float(v) for v in np.squeeze(clf.coef_)]

# join and sort in descending order of coefficient magnitude
arranged = sorted(
    [(label, coeff) for label, coeff in zip(labels, coeffs)],
    key=lambda p: abs(p[1]),
    reverse=True,
)

# exponentiate to get the odds increase, rather than log odds
arranged = [(label, math.exp(coeff)) for label, coeff in arranged]
arranged